# Importing the Dataset


> Nothing of note to explain here, code copy pasted directly from dataset website


In [ ]:
!pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17)

# data (as pandas dataframes)
X = breast_cancer_wisconsin_diagnostic.data.features
y = breast_cancer_wisconsin_diagnostic.data.targets

# metadata
print(breast_cancer_wisconsin_diagnostic.metadata)

# variable information
print(breast_cancer_wisconsin_diagnostic.variables)


{'uci_id': 17, 'name': 'Breast Cancer Wisconsin (Diagnostic)', 'repository_url': 'https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic', 'data_url': 'https://archive.ics.uci.edu/static/public/17/data.csv', 'abstract': 'Diagnostic Wisconsin Breast Cancer Database.', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 569, 'num_features': 30, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Diagnosis'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1993, 'last_updated': 'Fri Nov 03 2023', 'dataset_doi': '10.24432/C5DW2B', 'creators': ['William Wolberg', 'Olvi Mangasarian', 'Nick Street', 'W. Street'], 'intro_paper': {'ID': 230, 'type': 'NATIVE', 'title': 'Nuclear feature extraction for breast tumor diagnosis', 'authors': 'W. Street, W. Wolberg, O. Mangasarian', 'venue': 'Electronic imaging', 'year': 1993, 'journal': None, 'DOI': '1

In [ ]:
X.columns


Index(['radius1', 'texture1', 'perimeter1', 'area1', 'smoothness1',
       'compactness1', 'concavity1', 'concave_points1', 'symmetry1',
       'fractal_dimension1', 'radius2', 'texture2', 'perimeter2', 'area2',
       'smoothness2', 'compactness2', 'concavity2', 'concave_points2',
       'symmetry2', 'fractal_dimension2', 'radius3', 'texture3', 'perimeter3',
       'area3', 'smoothness3', 'compactness3', 'concavity3', 'concave_points3',
       'symmetry3', 'fractal_dimension3'],
      dtype='object')

# Preprocessing

In [ ]:
X = X[['perimeter3','area3','concavity3']] #Selecting a set number of features
X = (X - X.mean()) / X.std() # Standardizing so each feature is on a similar scale
y = y['Diagnosis'].map({'M': 1, 'B': 0}) # Assigning Binary Values

X.head()

,perimeter3,area3,concavity3
0,2.301575,1.999478,2.107672
1,1.533776,1.888827,-0.146620
2,1.346291,1.455004,0.854222
3,-0.249720,-0.549538,1.987839
4,1.337363,1.219651,0.612640


In [ ]:
X.shape[1]

3

# Importing NumPy

In [ ]:
import numpy as np

## Defining Functions

In [ ]:
hidden_size = 4
alpha = 0.01

In [ ]:
# initial weights
np.random.seed(42)
weights_01 = np.random.random((X.shape[1], hidden_size))
weights_12 = np.random.random((hidden_size, 1))
weights = [weights_01, weights_12]

# initial biases
bias_1 = np.zeros(hidden_size)   # hidden layer: (hidden_size,)
bias_2 = np.zeros(1)             # output layer: (1,)
biases = [bias_1, bias_2]

print(weights)
print(biases)


[array([[0.37454012, 0.95071431, 0.73199394, 0.59865848],
       [0.15601864, 0.15599452, 0.05808361, 0.86617615],
       [0.60111501, 0.70807258, 0.02058449, 0.96990985]]), array([[0.83244264],
       [0.21233911],
       [0.18182497],
       [0.18340451]])]
[array([0., 0., 0., 0.]), array([0.])]


## Defining Functions

In [ ]:

def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def binary_cross_entropy(pred, goal, eps=1e-12):
    pred = np.clip(pred, eps, 1 - eps)
    return -(goal * np.log(pred) + (1 - goal) * np.log(1 - pred)).sum()

def relu2deriv(output):
  return output > 0

def layer(input, weights, bias):
  return input.dot(weights) + bias



## Defining Neural Network

In [ ]:
def neural_network(input, weights, biases):
  hidden = relu(layer(input, weights[0], biases[0]))
  pred = sigmoid(layer(hidden, weights[1], biases[1]))  # probability of malignant
  return hidden, pred

## Testing


## Predict

In [ ]:
y_hidden, y_pred = neural_network(X.iloc[0], weights, biases)
print(y_pred)


[0.98463675]


In [ ]:
mse_error = (y_pred - y.iloc[0]) ** 2

bce_error = binary_cross_entropy(y_pred, y.iloc[0])
delta = (y_pred - y.iloc[0])

In [ ]:
print(f"MSE Error:  {mse_error[0]}")
print(f"BCE Error:  {bce_error}")

MSE Error:  0.00023602932071352454
BCE Error:  0.015482483260741307


## Compare and Learn

In [ ]:

X_np = X.values.astype(float)
y_np = y.values.astype(float).ravel()

alpha = 0.01

for epoch in range(100):
  total_error = 0
  for i in range(len(X_np)):
    input = X_np[i]                               # (3,)
    goal = y_np[i]

    # forward pass
    hidden, pred = neural_network(input, weights, biases)

    error = binary_cross_entropy(pred, goal)

    delta_2 = pred - goal                                       # (1,)
    delta_1 = delta_2.dot(weights[1].T) * relu2deriv(hidden)    # (hidden_size,)

    weights[1] = weights[1] - alpha * np.outer(hidden, delta_2) # (hidden_size, 1)
    weights[0] = weights[0] - alpha * np.outer(input, delta_1)  # (3, hidden_size)

    biases[1] = biases[1] - alpha * delta_2                     # (1,)
    biases[0] = biases[0] - alpha * delta_1                     # (hidden_size,)

    total_error += error
  if epoch % 10 == 0:
    print(f"Epoch {epoch}: mean loss = {total_error / len(X_np):.4f}")

print("Final weights:", weights)
print("Final biases:", biases)



Epoch 0: mean loss = 0.4053
Epoch 10: mean loss = 0.1564
Epoch 20: mean loss = 0.1491
Epoch 30: mean loss = 0.1475
Epoch 40: mean loss = 0.1466
Epoch 50: mean loss = 0.1455
Epoch 60: mean loss = 0.1449
Epoch 70: mean loss = 0.1446
Epoch 80: mean loss = 0.1444
Epoch 90: mean loss = 0.1442
Final weights: [array([[0.79333492, 1.09556911, 0.76287846, 0.4840757 ],
       [1.2320827 , 0.67215732, 0.38763216, 1.2264242 ],
       [0.34851779, 0.30255791, 0.18347738, 0.72475405]]), array([[1.40559169],
       [0.74481106],
       [0.53890021],
       [1.25328525]])]
Final biases: [array([0.76406891, 0.65499682, 0.4216262 , 1.27847591]), array([-4.35541427])]


# Checklist
## Week 1
- [X] Import Dataset
- [X] Preprocess
- [X] Build Neural Network
- [X] Predict, Compare and Learn
- [X] Gradient Descent
- [X] Activation Functions
## Week 2
- [x] Backpropagation
## Week 3
- [x] Adding Bias
- [x] Different Loss Functions
## Week 4
- [ ] Optimizers
## Week 5
- [ ] Neural Network using Framework
- [ ] Model Evaluation
## Week 6
- [ ] Transformers